# EURUSD Trading Strategy

Processing EURUSD OHLCV data from local files at 1-hour (H1) and 5-minute (M5) intervals

# Trading Strategy Configuration

Define all configurable parameters for the trading strategy

In [ ]:
# Trading Configuration
CONFIG = {
    # Data Parameters
    'symbol': 'EURUSD',
    'exchange': 'OANDA',
    'timeframes': ['H1', 'M5'],
    'history_days': 7,
    'data_directory': 'data',
    
    # Trading Parameters
    'initial_balance': 10.0,
    'position_size_percent': 2.0,
    'pip_value': 0.0001,
    'stop_loss_pips': 10,
    
    # Indicator Parameters
    'trend_ema_period': 8,
    'entry_ema_period': 5,
    'rsi_period': 9,
    'rsi_overbought': 70,
    'rsi_oversold': 30,
    'macd_fast': 8,
    'macd_slow': 17,
    'macd_signal': 9,
    
    # Session Parameters
    'sessions': {
        'London': {'start': 8, 'end': 16},
        'NewYork': {'start': 13, 'end': 21},
        'Overlap': {'start': 13, 'end': 16}
    },
    
    # Output Parameters
    'output_directory': './strategy_results'
}

# Create necessary directories
import os
os.makedirs(CONFIG['data_directory'], exist_ok=True)
os.makedirs(CONFIG['output_directory'], exist_ok=True)

In [ ]:
import pandas as pd
from datetime import datetime, timedelta
import time
import os
import json
import matplotlib.pyplot as plt
import pandas_ta as ta
import numpy as np
import schedule
from tvDatafeed import TvDatafeed, Interval
import re
import logging

In [4]:
# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler("data_fetcher.log"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

class TradingViewDataFetcher:
    def __init__(self, username=None, password=None):
        """Initialize the TradingView data fetcher with optional credentials."""
        logger.info("Connecting to TradingView...")
        try:
            if username and password:
                self.tv = TvDatafeed(username=username, password=password)
                logger.info("Connection established with credentials!")
            else:
                self.tv = TvDatafeed()
                logger.info("Connection established without login (some data may be limited)")
            
            # Ensure output directory exists
            os.makedirs('data', exist_ok=True)
            
        except Exception as e:
            logger.error(f"Failed to connect to TradingView: {e}")
            raise
        
    def fetch_historical_data(self, symbol, exchange, interval, days_back=7):
        """Fetch historical data for specified interval and days back."""
        logger.info(f"Fetching historical {interval} data for {symbol}...")
        
        try:
            # Calculate exact timestamps
            end_time = datetime.now()
            start_time = end_time - timedelta(days=days_back)
            
            # Map our interval names to tvDatafeed Interval objects
            interval_map = {
                'H1': Interval.in_1_hour,
                'M5': Interval.in_5_minute
            }
            
            # Calculate how many bars we need based on interval
            if interval == 'H1':
                n_bars = int((end_time - start_time).total_seconds() / 3600) + 10  # Add buffer
            elif interval == 'M5':
                n_bars = int((end_time - start_time).total_seconds() / 300) + 10  # Add buffer
            
            logger.debug(f"Requesting {n_bars} bars for {interval}")
            
            # Fetch the data
            df = self.tv.get_hist(
                symbol=symbol,
                exchange=exchange,
                interval=interval_map[interval],
                n_bars=n_bars
            )
            
            if df is None or df.empty:
                logger.warning(f"No data returned for {symbol} on {exchange} with interval {interval}")
                return None
                
            # Reset index to make datetime a column and format for CSV
            df = df.reset_index()
            
            # Save to CSV with simplified naming
            filepath = os.path.join('data', f'{interval}.csv')
            df.to_csv(filepath, index=False)
            logger.info(f"Saved historical data to {filepath} ({len(df)} rows)")
            
            return df
            
        except Exception as e:
            logger.error(f"Error fetching historical data: {e}")
            raise
    
    def is_candle_complete(self, interval, datetime_val):
        """Check if the current candle is complete based on its datetime"""
        try:
            current_time = datetime.now()
            
            # Convert to datetime if string
            if isinstance(datetime_val, str):
                candle_time = pd.to_datetime(datetime_val)
            else:
                candle_time = datetime_val
                
            # Remove timezone info if present to avoid comparison issues
            if hasattr(candle_time, 'tz_localize'):
                candle_time = candle_time.tz_localize(None)
            
            # Calculate when the next candle should start
            if interval == 'H1':
                next_candle = candle_time + pd.Timedelta(hours=1)
            elif interval == 'M5':
                next_candle = candle_time + pd.Timedelta(minutes=5)
            else:
                logger.warning(f"Unknown interval: {interval}")
                return False

            # Debug information
            logger.debug(f"Candle time: {candle_time}, Next candle: {next_candle}, Current time: {current_time}")
            is_complete = current_time >= next_candle
            
            if is_complete:
                logger.debug(f"Candle at {candle_time} is complete")
            else:
                logger.debug(f"Candle at {candle_time} is still forming ({(next_candle - current_time).total_seconds():.1f} seconds remaining)")
                
            return is_complete
            
        except Exception as e:
            logger.error(f"Error checking if candle is complete: {e}")
            return False

    def update_data(self, symbol, exchange, interval):
        """Update data for the specified interval"""
        logger.info(f"Updating {interval} data for {symbol}...")

        try:
            # Map intervals to tvDatafeed Interval objects
            interval_map = {
                'H1': Interval.in_1_hour,
                'M5': Interval.in_5_minute
            }

            # Get the latest data
            latest_df = self.tv.get_hist(
                symbol=symbol,
                exchange=exchange,
                interval=interval_map[interval],
                n_bars=10  # Increased to get more historical bars
            )
            
            if latest_df is None or latest_df.empty:
                logger.warning(f"No data returned when updating {interval} data")
                return None

            latest_df = latest_df.reset_index()
            
            # Log the first and last data timestamps
            if not latest_df.empty:
                first_time = latest_df['datetime'].iloc[0]
                last_time = latest_df['datetime'].iloc[-1]
                logger.debug(f"Data range: {first_time} to {last_time} ({len(latest_df)} rows)")

            # Filter out incomplete candles
            complete_df = latest_df[latest_df['datetime'].apply(
                lambda x: self.is_candle_complete(interval, x)
            )]

            if complete_df.empty:
                candles_checked = min(5, len(latest_df))
                candle_times = latest_df['datetime'].head(candles_checked).tolist()
                logger.info(f"No complete candles available for {interval}. Latest candles: {candle_times}")
                return None

            logger.info(f"Found {len(complete_df)} complete candles for {interval}")

            filepath = os.path.join('data', f'{interval}.csv')
            if os.path.exists(filepath):
                existing_df = pd.read_csv(filepath)
                existing_df['datetime'] = pd.to_datetime(existing_df['datetime'])

                # Check for new data
                new_data = complete_df[~complete_df['datetime'].isin(existing_df['datetime'])]
                if new_data.empty:
                    logger.info(f"No new {interval} data to update")
                    return existing_df
                
                logger.info(f"Found {len(new_data)} new {interval} candles to add")
                
                # Remove any overlapping data and append new complete data
                existing_df = existing_df[~existing_df['datetime'].isin(complete_df['datetime'])]
                combined_df = pd.concat([existing_df, complete_df]).sort_values('datetime')

                combined_df.to_csv(filepath, index=False)
                logger.info(f"Updated {interval} data in {filepath} (now {len(combined_df)} rows)")
                return combined_df
            else:
                complete_df.to_csv(filepath, index=False)
                logger.info(f"Created new file {filepath} with {interval} data ({len(complete_df)} rows)")
                return complete_df
                
        except Exception as e:
            logger.error(f"Error updating {interval} data: {e}")
            return None

if __name__ == "__main__":
    try:
        logger.info("Starting TradingView Data Fetcher")
        
        # Get login credentials
        use_credentials = input("Do you want to use TradingView credentials? (y/n): ").strip().lower() == 'y'
        username = None
        password = None
        
        if use_credentials:
            username = input("Enter your TradingView username: ")
            password = input("Enter your TradingView password: ")
            logger.info("Credentials provided, attempting authenticated connection")
        
        # Initialize the fetcher with credentials if provided
        fetcher = TradingViewDataFetcher(username, password)
        
        # Get symbol and exchange from input
        symbol = input("Enter the symbol to track (e.g., BTCUSDT): ")
        exchange = input("Enter the exchange (e.g., BINANCE): ")
        
        # First fetch historical data
        h1_data = fetcher.fetch_historical_data(symbol, exchange, 'H1', days_back=7)
        m5_data = fetcher.fetch_historical_data(symbol, exchange, 'M5', days_back=7)
        
        if h1_data is None and m5_data is None:
            logger.error("Failed to fetch initial historical data. Please check connection and retry.")
            exit(1)
        
        logger.info("\nData fetcher is now running with 15-second updates. Press Ctrl+C to stop.")
        
        update_count = 0
        last_successful_h1 = None
        last_successful_m5 = None
        
        # Run continuous updates
        while True:
            try:
                h1_updated = fetcher.update_data(symbol, exchange, 'H1')
                if h1_updated is not None:
                    last_successful_h1 = datetime.now()
                    logger.info(f"Successfully updated H1 data")
                
                m5_updated = fetcher.update_data(symbol, exchange, 'M5')
                if m5_updated is not None:
                    last_successful_m5 = datetime.now()
                    logger.info(f"Successfully updated M5 data")
                
                update_count += 1
                if update_count % 20 == 0:  # Log every 20 updates (approx. 5 minutes)
                    logger.info(f"Fetcher running for {update_count} update cycles")
                    
                    # Report time since last successful updates
                    current_time = datetime.now()
                    if last_successful_h1:
                        time_since_h1 = current_time - last_successful_h1
                        logger.info(f"Time since last successful H1 update: {time_since_h1}")
                    if last_successful_m5:
                        time_since_m5 = current_time - last_successful_m5
                        logger.info(f"Time since last successful M5 update: {time_since_m5}")
                
                time.sleep(15)  # Wait for 15 seconds before next update
                
            except KeyboardInterrupt:
                logger.info("\nData fetcher stopped by user.")
                break
            except Exception as e:
                logger.error(f"Error during update: {e}")
                logger.info("Will retry in 15 seconds...")
                time.sleep(15)  # Wait before retrying
                
    except Exception as e:
        logger.critical(f"Critical error in main execution: {e}")
        exit(1)

2025-04-28 15:30:09,149 - INFO - Starting TradingView Data Fetcher
2025-04-28 15:30:29,199 - INFO - Credentials provided, attempting authenticated connection
2025-04-28 15:30:29,213 - INFO - Connecting to TradingView...
2025-04-28 15:30:29,832 - ERROR - error while signin
2025-04-28 15:30:29,834 - WARNING - you are using nologin method, data you access may be limited
2025-04-28 15:30:29,842 - INFO - Connection established with credentials!
2025-04-28 15:30:39,833 - INFO - Fetching historical H1 data for XAUUSD...
2025-04-28 15:30:41,450 - INFO - Saved historical data to data/H1.csv (178 rows)
2025-04-28 15:30:41,451 - INFO - Fetching historical M5 data for XAUUSD...
2025-04-28 15:30:44,176 - INFO - Saved historical data to data/M5.csv (2026 rows)
2025-04-28 15:30:44,177 - INFO - 
Data fetcher is now running with 15-second updates. Press Ctrl+C to stop.
2025-04-28 15:30:44,179 - INFO - Updating H1 data for XAUUSD...
2025-04-28 15:30:45,600 - INFO - Found 9 complete candles for H1
2025-0

## Real-Time Data Cleaning

Process and clean data as new complete candles are received

In [5]:
class DataProcessor:
    def __init__(self, base_path='.'):
        """Initialize the data processor with base path for files"""
        self.base_path = base_path
        self.last_processed = {
            'H1': None,
            'M5': None
        }

    def is_candle_complete(self, row, timeframe):
        """Check if a candle is complete based on its timestamp"""
        current_time = pd.Timestamp.now(tz='UTC')
        candle_time = pd.to_datetime(row['datetime']).tz_localize('UTC')

        if timeframe == 'H1':
            next_candle = candle_time + pd.Timedelta(hours=1)
        elif timeframe == 'M5':
            next_candle = candle_time + pd.Timedelta(minutes=5)

        return current_time >= next_candle

    def clean_data(self, df, symbol):
        """Clean and standardize the dataframe"""
        df = df.copy()

        # Ensure datetime is properly formatted
        df['datetime'] = pd.to_datetime(df['datetime'])

        # Standardize column names
        column_mapping = {
            'c': 'close',
            'o': 'open',
            'h': 'high',
            'l': 'low',
            'v': 'volume'
        }

        for old, new in column_mapping.items():
            if old in df.columns and new not in df.columns:
                df[new] = df[old]

        # Add symbol if not present
        if 'symbol' not in df.columns:
            df['symbol'] = symbol

        # Remove any duplicate timestamps
        df = df.drop_duplicates(subset=['datetime'])

        # Sort by datetime
        df = df.sort_values('datetime')

        return df

    def process_new_data(self, timeframe, symbol='EURUSD'):
        """Process new data from CSV files when complete candles are available"""
        file_path = f'{timeframe}.csv'
        cleaned_file_path = f'{timeframe}_cleaned.csv'

        if not os.path.exists(file_path):
            print(f"No data file found for {timeframe}")
            return None

        # Read the raw data
        df = pd.read_csv(file_path)
        
        # Filter for complete candles only
        complete_mask = df.apply(lambda row: self.is_candle_complete(row, timeframe), axis=1)
        complete_df = df[complete_mask]

        if complete_df.empty:
            print(f"No complete candles found for {timeframe}")
            return None

        # Get last processed timestamp
        last_processed = self.last_processed[timeframe]

        if last_processed:
            # Only process new data since last processing
            new_data = complete_df[pd.to_datetime(complete_df['datetime']) > last_processed]
            if new_data.empty:
                print(f"No new complete candles for {timeframe}")
                return None
        else:
            new_data = complete_df

        # Clean the new data
        cleaned_data = self.clean_data(new_data, symbol)

        # Update or create the cleaned file
        if os.path.exists(cleaned_file_path):
            existing_cleaned = pd.read_csv(cleaned_file_path)
            existing_cleaned['datetime'] = pd.to_datetime(existing_cleaned['datetime'])
            # Combine existing and new data, removing any duplicates
            combined = pd.concat([existing_cleaned, cleaned_data])
            combined = combined.drop_duplicates(subset=['datetime']).sort_values('datetime')
            combined.to_csv(cleaned_file_path, index=False)
        else:
            cleaned_data.to_csv(cleaned_file_path, index=False)

        # Update last processed timestamp
        self.last_processed[timeframe] = pd.to_datetime(cleaned_data['datetime']).max()

        return cleaned_data

    def get_latest_data(self):
        """Process new data for all timeframes and return the latest clean data"""
        results = {}
        for timeframe in ['H1', 'M5']:
            cleaned_data = self.process_new_data(timeframe)
            if cleaned_data is not None:
                print(f"Processed {len(cleaned_data)} new candles for {timeframe}")
                results[timeframe] = cleaned_data
        return results

## Load and Process Data

Load data from local CSV files and prepare it for analysis

In [6]:
# Load the data
h1_data = pd.read_csv('H1_cleaned.csv')
m5_data = pd.read_csv('M5_cleaned.csv')

# Convert datetime columns if needed
if 'datetime' in h1_data.columns or 'time' in h1_data.columns:
    datetime_col = 'datetime' if 'datetime' in h1_data.columns else 'time'
    h1_data[datetime_col] = pd.to_datetime(h1_data[datetime_col])
    h1_data.set_index(datetime_col, inplace=True)

if 'datetime' in m5_data.columns or 'time' in m5_data.columns:
    datetime_col = 'datetime' if 'datetime' in m5_data.columns else 'time'
    m5_data[datetime_col] = pd.to_datetime(m5_data[datetime_col])
    m5_data.set_index(datetime_col, inplace=True)

# Check and rename columns if necessary
# OANDA typically uses 'c' for close, 'o' for open, etc.
column_mapping = {
    'c': 'close', 'o': 'open', 'h': 'high', 'l': 'low', 'v': 'volume'
}

for old, new in column_mapping.items():
    if old in h1_data.columns and new not in h1_data.columns:
        h1_data[new] = h1_data[old]
    if old in m5_data.columns and new not in m5_data.columns:
        m5_data[new] = m5_data[old]

# Calculate 1H indicators
print("Calculating 1H indicators...")
# Trend Bias: 8-period EMA on 1-hour chart
h1_data['ema_8'] = ta.ema(h1_data['close'], length=8)
h1_data['trend_bias'] = np.where(h1_data['close'] > h1_data['ema_8'], 'bullish', 'bearish')

# Calculate 5M indicators
print("Calculating 5M indicators...")
# Entry Points: 5-period EMA on 5-minute chart
m5_data['ema_5'] = ta.ema(m5_data['close'], length=5)
m5_data['price_crossed_above_ema'] = np.where(
    (m5_data['close'] > m5_data['ema_5']) & (m5_data['close'].shift(1) <= m5_data['ema_5'].shift(1)), 
    True, False
)
m5_data['price_crossed_below_ema'] = np.where(
    (m5_data['close'] < m5_data['ema_5']) & (m5_data['close'].shift(1) >= m5_data['ema_5'].shift(1)), 
    True, False
)

# RSI indicator for divergence (alternative entry)
m5_data['rsi_9'] = ta.rsi(m5_data['close'], length=9)

# Exit trades: MACD (8, 17, 9) on 5-minute chart
macd = ta.macd(m5_data['close'], fast=8, slow=17, signal=9)
m5_data = m5_data.join(macd)

# Identify MACD crossovers
m5_data['macd_cross_below'] = np.where(
    (m5_data['MACD_8_17_9'] < m5_data['MACDs_8_17_9']) & 
    (m5_data['MACD_8_17_9'].shift(1) >= m5_data['MACDs_8_17_9'].shift(1)), 
    True, False
)
m5_data['macd_cross_above'] = np.where(
    (m5_data['MACD_8_17_9'] > m5_data['MACDs_8_17_9']) & 
    (m5_data['MACD_8_17_9'].shift(1) <= m5_data['MACDs_8_17_9'].shift(1)), 
    True, False
)

# Save the results
h1_data.to_csv('H1_with_indicators.csv')
m5_data.to_csv('M5_with_indicators.csv')

print("1H Data Sample:")
print(h1_data[['close', 'ema_8', 'trend_bias']].tail())

print("\n5M Data Sample:")
print(m5_data[['close', 'ema_5', 'rsi_9', 'MACD_8_17_9', 'MACDs_8_17_9', 
               'macd_cross_above', 'macd_cross_below']].tail())

print("Indicators calculated and saved to CSV files!")

# Create signals DataFrame
signals_df = pd.DataFrame({
    'datetime': m5_data.index,
    'close': m5_data['close'],
    'ema_5': m5_data['ema_5'],
    'rsi_9': m5_data['rsi_9'],
    'macd': m5_data['MACD_8_17_9'],
    'macd_signal': m5_data['MACDs_8_17_9'],
    'price_crossed_above_ema': m5_data['price_crossed_above_ema'],
    'price_crossed_below_ema': m5_data['price_crossed_below_ema'],
    'macd_cross_above': m5_data['macd_cross_above'],
    'macd_cross_below': m5_data['macd_cross_below']
})

# Add hourly trend bias by resampling to 5-minute timeframe
hourly_bias = h1_data['trend_bias'].reindex(m5_data.index, method='ffill')
signals_df['trend_bias'] = hourly_bias

signals_df

# Function to visualize the indicators for a specific time period (optional)
def visualize_indicators(h1_sample, m5_sample, period=50):
    """
    Visualize the indicators for analysis
    """
    # 1H Chart with EMA
    plt.figure(figsize=(14, 7))
    plt.subplot(2, 1, 1)
    plt.title('1H Chart - Trend Bias')
    plt.plot(h1_sample.index[-period:], h1_sample['close'][-period:], label='Close')
    plt.plot(h1_sample.index[-period:], h1_sample['ema_8'][-period:], label='EMA(8)')
    plt.legend()
    plt.grid(True)
    
    # 5M Chart with Entry/Exit signals
    plt.subplot(2, 1, 2)
    plt.title('5M Chart - Entry & Exit Points')
    plt.plot(m5_sample.index[-period*12:], m5_sample['close'][-period*12:], label='Close')
    plt.plot(m5_sample.index[-period*12:], m5_sample['ema_5'][-period*12:], label='EMA(5)')
    
    # Plot entry signals
    entries_long = m5_sample[-period*12:][m5_sample['price_crossed_above_ema'][-period*12:]]
    entries_short = m5_sample[-period*12:][m5_sample['price_crossed_below_ema'][-period*12:]]
    plt.scatter(entries_long.index, entries_long['close'], color='green', marker='^', s=100, label='Long Entry')
    plt.scatter(entries_short.index, entries_short['close'], color='red', marker='v', s=100, label='Short Entry')
    
    # Plot exit signals
    exits_long = m5_sample[-period*12:][m5_sample['macd_cross_below'][-period*12:]]
    exits_short = m5_sample[-period*12:][m5_sample['macd_cross_above'][-period*12:]]
    plt.scatter(exits_long.index, exits_long['close'], color='orange', marker='x', s=100, label='Long Exit')
    plt.scatter(exits_short.index, exits_short['close'], color='purple', marker='x', s=100, label='Short Exit')
    
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig('indicator_analysis.png')
    plt.show()

# Uncomment to visualize (if you want to see the indicators graphically)
# visualize_indicators(h1_data, m5_data)

print("Indicators calculated and saved to CSV files!")

Calculating 1H indicators...
Calculating 5M indicators...
1H Data Sample:
                              close        ema_8 trend_bias
time                                                       
2025-04-28 11:00:00+00:00  3282.790  3289.570132    bearish
2025-04-28 12:00:00+00:00  3274.605  3286.244547    bearish
2025-04-28 13:00:00+00:00  3289.630  3286.996870    bullish
2025-04-28 14:00:00+00:00  3294.285  3288.616454    bullish
2025-04-28 15:00:00+00:00  3299.595  3291.056131    bullish

5M Data Sample:
                              close        ema_5      rsi_9  MACD_8_17_9  \
time                                                                       
2025-04-28 15:15:00+00:00  3295.860  3295.315629  71.038972     2.325948   
2025-04-28 15:20:00+00:00  3299.280  3296.637086  79.183122     2.597774   
2025-04-28 15:25:00+00:00  3302.115  3298.463057  83.508085     3.036560   
2025-04-28 15:30:00+00:00  3299.135  3298.687038  67.037765     2.933831   
2025-04-28 15:35:00+00:00  3301.0

## Trading Sessions

Define London and New York trading sessions for filtering trades:
- London: 08:00-16:00 UTC
- New York: 13:00-21:00 UTC
- Overlap: 13:00-16:00 UTC (prime trading hours)

In [9]:
def is_trading_session(timestamp):
    """Check if timestamp is within London or New York trading sessions"""
    hour = timestamp.hour
    # Convert to UTC if needed
    if hasattr(timestamp, 'tz_localize'):
        timestamp = timestamp.tz_localize(None)
    # London session: 8:00-16:00 UTC
    # New York session: 13:00-21:00 UTC
    # Only allow trades during these specific hours
    return 8 <= hour < 21 and hour not in [16, 17]  # Exclude less liquid hours

def get_session_label(timestamp):
    """Return the trading session label for a given timestamp"""
    if not is_trading_session(timestamp):
        return 'Off Hours'
    
    hour = timestamp.hour
    if 8 <= hour < 13:
        return 'London'
    elif 13 <= hour < 16:
        return 'Overlap'
    elif 18 <= hour < 21:
        return 'New York'
    return 'Off Hours'

# Add session information to signals DataFrame
signals_df['session'] = signals_df.index.map(get_session_label)
signals_df['is_trading_hours'] = signals_df.index.map(is_trading_session)

# Display session distribution
print("Trading Session Distribution:")
print(signals_df['session'].value_counts())

Trading Session Distribution:
session
Off Hours    1011
London        480
Overlap       284
New York      252
Name: count, dtype: int64


In [8]:
# Reset existing signals
signals_df['signal'] = 'no_signal'

# Generate signals ONLY during trading hours
mask = signals_df['is_trading_hours']

# Long entry signals during trading hours
signals_df.loc[mask & 
              (signals_df['trend_bias'] == 'bullish') & 
              (signals_df['price_crossed_above_ema']) & 
              (signals_df['rsi_9'] > 30) & 
              (signals_df['session'].isin(['London', 'Overlap', 'New York'])), 'signal'] = 'buy'

# Short entry signals during trading hours
signals_df.loc[mask & 
              (signals_df['trend_bias'] == 'bearish') & 
              (signals_df['price_crossed_below_ema']) & 
              (signals_df['rsi_9'] < 70) & 
              (signals_df['session'].isin(['London', 'Overlap', 'New York'])), 'signal'] = 'sell'

# Exit signals (still allowing exits during all hours for risk management)
signals_df.loc[signals_df['macd_cross_below'], 'signal'] = 'exit_buy'
signals_df.loc[signals_df['macd_cross_above'], 'signal'] = 'exit_sell'

# Calculate stop losses (20 pips)
pip_value = 0.0001
stop_loss_pips = 20
signals_df['stop_loss_long'] = signals_df['close'] - (stop_loss_pips * pip_value)
signals_df['stop_loss_short'] = signals_df['close'] + (stop_loss_pips * pip_value)

# Print signal distribution by session
print("\nSignal Distribution by Session:")
print(pd.crosstab(signals_df['session'], signals_df['signal']))


Signal Distribution by Session:
signal     buy  exit_buy  exit_sell  no_signal  sell
session                                             
London      22        23         19        380    36
New York    15        11         12        202    12
Off Hours    0        43         44        924     0
Overlap     22        17         20        207    18


## Trading Env

In [10]:
def calculate_performance_metrics(trades_df, portfolio_df, initial_balance):
    """
    Calculate various performance metrics based on trade history.
    """
    if trades_df is None or trades_df.empty:
        return {"error": "No trades executed"}

    # Copy dataframes to avoid modifying originals
    trades = trades_df.copy()
    portfolio = portfolio_df.copy()
    
    # Ensure datetime index
    if not isinstance(trades.index, pd.DatetimeIndex):
        trades.index = pd.to_datetime(trades.index)
    
    # Basic metrics
    total_trades = len(trades)
    winning_trades = len(trades[trades['pnl'] > 0])
    losing_trades = len(trades[trades['pnl'] < 0])
    win_rate = (winning_trades / total_trades * 100) if total_trades > 0 else 0
    
    # PnL metrics
    total_profit = trades[trades['pnl'] > 0]['pnl'].sum()
    total_loss = abs(trades[trades['pnl'] < 0]['pnl'].sum())
    net_profit = total_profit - total_loss
    profit_factor = total_profit / total_loss if total_loss > 0 else float('inf')
    
    # Average metrics
    avg_profit = trades[trades['pnl'] > 0]['pnl'].mean() if winning_trades > 0 else 0
    avg_loss = trades[trades['pnl'] < 0]['pnl'].mean() if losing_trades > 0 else 0
    avg_trade = trades['pnl'].mean()
    
    # Risk metrics
    risk_reward_ratio = abs(avg_profit / avg_loss) if avg_loss != 0 else float('inf')
    
    # Return metrics
    total_return = (portfolio['equity'].iloc[-1] / initial_balance - 1) * 100
    
    # Duration metrics - using vectorized datetime operations
    trades['duration'] = (pd.to_datetime(trades['exit_time']).dt.tz_localize(None) - 
                          trades.index.tz_localize(None)).dt.total_seconds() / 3600  # hours
    avg_trade_duration = trades['duration'].mean()
    
    # Maximum drawdown
    portfolio['cummax'] = portfolio['equity'].cummax()
    portfolio['drawdown'] = (portfolio['equity'] - portfolio['cummax']) / portfolio['cummax'] * 100
    max_drawdown = abs(portfolio['drawdown'].min())
    
    # Stop loss metrics
    stop_loss_triggered = len(trades[trades['exit_reason'] == 'stop_loss'])
    stop_loss_percentage = (stop_loss_triggered / total_trades * 100) if total_trades > 0 else 0
    
    # Position metrics
    long_trades = len(trades[trades['position'] == 'long'])
    short_trades = len(trades[trades['position'] == 'short'])
    long_win_rate = (len(trades[(trades['position'] == 'long') & (trades['pnl'] > 0)]) / long_trades * 100) if long_trades > 0 else 0
    short_win_rate = (len(trades[(trades['position'] == 'short') & (trades['pnl'] > 0)]) / short_trades * 100) if short_trades > 0 else 0
    
    # Sharpe ratio (simplified, assuming risk-free rate of 0)
    daily_returns = portfolio['equity'].pct_change().dropna()
    sharpe_ratio = (daily_returns.mean() / daily_returns.std()) * (252 ** 0.5) if len(daily_returns) > 0 and daily_returns.std() > 0 else 0
    
    return {
        "total_trades": total_trades,
        "winning_trades": winning_trades,
        "losing_trades": losing_trades,
        "win_rate": win_rate,
        "total_profit": float(total_profit),
        "total_loss": float(total_loss),
        "net_profit": float(net_profit),
        "profit_factor": float(profit_factor),
        "avg_profit": float(avg_profit),
        "avg_loss": float(avg_loss),
        "avg_trade": float(avg_trade),
        "risk_reward_ratio": float(risk_reward_ratio),
        "total_return_percent": float(total_return),
        "avg_trade_duration_hours": float(avg_trade_duration),
        "max_drawdown_percent": float(max_drawdown),
        "stop_loss_triggered": stop_loss_triggered,
        "stop_loss_percentage": float(stop_loss_percentage),
        "long_trades": long_trades,
        "short_trades": short_trades,
        "long_win_rate": float(long_win_rate),
        "short_win_rate": float(short_win_rate),
        "sharpe_ratio": float(sharpe_ratio),
        "initial_balance": float(initial_balance),
        "final_balance": float(portfolio['equity'].iloc[-1])
    }

def generate_performance_charts(trades_df, portfolio_df, metrics, output_dir, timestamp_str):
    """
    Generate and save performance charts
    """
    import matplotlib.pyplot as plt
    import os
    
    # Create charts directory
    charts_dir = os.path.join(output_dir, f'charts_{timestamp_str}')
    os.makedirs(charts_dir, exist_ok=True)
    
    # 1. Equity curve
    plt.figure(figsize=(10, 6))
    portfolio_df['equity'].plot()
    plt.title('Equity Curve')
    plt.ylabel('Equity ($)')
    plt.grid(True)
    plt.savefig(os.path.join(charts_dir, 'equity_curve.png'))
    plt.close()
    
    # 2. Drawdown chart
    plt.figure(figsize=(10, 6))
    portfolio_df['cummax'] = portfolio_df['equity'].cummax()
    portfolio_df['drawdown'] = (portfolio_df['equity'] - portfolio_df['cummax']) / portfolio_df['cummax'] * 100
    portfolio_df['drawdown'].plot()
    plt.title('Drawdown (%)')
    plt.ylabel('Drawdown (%)')
    plt.grid(True)
    plt.savefig(os.path.join(charts_dir, 'drawdown.png'))
    plt.close()
    
    # 3. Trade PnL distribution
    if not trades_df.empty:
        plt.figure(figsize=(10, 6))
        trades_df['pnl'].hist(bins=20)
        plt.title('Trade P&L Distribution')
        plt.xlabel('P&L ($)')
        plt.ylabel('Frequency')
        plt.grid(True)
        plt.savefig(os.path.join(charts_dir, 'pnl_distribution.png'))
        plt.close()
    
    # 4. Win/Loss pie chart
    if metrics['total_trades'] > 0:
        plt.figure(figsize=(8, 8))
        plt.pie([metrics['winning_trades'], metrics['losing_trades']], 
                labels=['Winning', 'Losing'], 
                autopct='%1.1f%%', 
                colors=['green', 'red'])
        plt.title('Win/Loss Ratio')
        plt.savefig(os.path.join(charts_dir, 'win_loss_ratio.png'))
        plt.close()
    
    # 5. Trade duration vs P&L scatter plot
    if not trades_df.empty and 'duration' in trades_df.columns:
        plt.figure(figsize=(10, 6))
        colors = ['green' if pnl > 0 else 'red' for pnl in trades_df['pnl']]
        plt.scatter(trades_df['duration'], trades_df['pnl'], c=colors, alpha=0.7)
        plt.title('Trade Duration vs P&L')
        plt.xlabel('Duration (hours)')
        plt.ylabel('P&L ($)')
        plt.grid(True)
        plt.savefig(os.path.join(charts_dir, 'duration_vs_pnl.png'))
        plt.close()
    
    # 6. Cumulative P&L chart
    if not trades_df.empty:
        plt.figure(figsize=(10, 6))
        trades_df['pnl'].cumsum().plot()
        plt.title('Cumulative P&L')
        plt.ylabel('Cumulative P&L ($)')
        plt.grid(True)
        plt.savefig(os.path.join(charts_dir, 'cumulative_pnl.png'))
        plt.close()
    
    print(f"Performance charts saved to {charts_dir}")
    
    


In [11]:
def execute_trades(signals_df, initial_balance=10000, position_size_percent=2, output_dir='./strategy_results'):
    """
    Execute trades based on generated signals, track performance, and save results to files.
    Added warmup period of 5 days and trade logging to CSV.
    """
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    print(f"📁 Output directory created/verified: {output_dir}")
    
    # Calculate warmup period (5 days)
    start_date = signals_df.index[0].date()
    trading_start_date = start_date + timedelta(days=5)
    print(f"Warmup period: {start_date} to {trading_start_date}")
    
    # Initialize trade tracking
    trades = []
    balance = initial_balance
    position = None
    entry_price = None
    entry_time = None
    stop_loss = None

    # Portfolio history
    portfolio_history = [{'timestamp': signals_df.index[0], 'balance': balance, 'equity': balance}]
    
    # Progress tracking
    total_rows = len(signals_df)
    print(f"🔄 Processing {total_rows} price data points...")
    
    # Iterate through the signals
    for i, (timestamp, row) in enumerate(signals_df.iterrows()):
        current_price = row['close']
        current_date = timestamp.date()

        # Skip all trading activity during warmup period
        if current_date <= trading_start_date:
            portfolio_history.append({
                'timestamp': timestamp,
                'balance': balance,
                'equity': balance
            })
            continue

        # Handle existing position
        if position:
            # Check for position exit conditions
            exit_price = None
            exit_reason = None

            # Check stop loss
            if (position == 'long' and current_price <= stop_loss) or \
               (position == 'short' and current_price >= stop_loss):
                exit_price = stop_loss
                exit_reason = 'stop_loss'

            # Check exit signals
            elif ((position == 'long' and row['signal'] == 'exit_buy') or
                  (position == 'short' and row['signal'] == 'exit_sell')):
                exit_price = current_price
                exit_reason = 'signal'

            # Execute exit if conditions met
            if exit_price is not None:
                trade_pnl = (exit_price - entry_price) if position == 'long' else (entry_price - exit_price)
                trade_pnl *= position_size
                balance += trade_pnl

                trades.append({
                    'entry_time': entry_time,
                    'entry_price': entry_price,
                    'position': position,
                    'exit_time': timestamp,
                    'exit_price': exit_price,
                    'stop_loss': stop_loss,
                    'pnl': trade_pnl,
                    'pnl_percent': (trade_pnl / (position_size * entry_price)) * 100,
                    'exit_reason': exit_reason,
                    'win_loss': 'win' if trade_pnl > 0 else 'loss',
                    'monetary_change': trade_pnl,
                    'balance_before': balance - trade_pnl,
                    'balance_after': balance
                })

                position = None
                entry_price = None
                stop_loss = None

        # Enter new position if no current position and after warmup period
        elif row['is_trading_hours']:
            if row['signal'] == 'buy':
                position = 'long'
                entry_price = current_price
                entry_time = timestamp
                stop_loss = row['stop_loss_long']
                position_size = (balance * position_size_percent / 100) / (entry_price - stop_loss)

            elif row['signal'] == 'sell':
                position = 'short'
                entry_price = current_price
                entry_time = timestamp
                stop_loss = row['stop_loss_short']
                position_size = (balance * position_size_percent / 100) / (stop_loss - entry_price)

        # Update portfolio value
        equity = balance
        if position:
            unrealized_pnl = ((current_price - entry_price) if position == 'long' else 
                             (entry_price - current_price)) * position_size
            equity += unrealized_pnl

        portfolio_history.append({
            'timestamp': timestamp,
            'balance': balance,
            'equity': equity
        })

    # Close any open position at the end
    if position:
        trade_pnl = ((current_price - entry_price) if position == 'long' else 
                    (entry_price - current_price)) * position_size
        balance += trade_pnl

        trades.append({
            'entry_time': entry_time,
            'entry_price': entry_price,
            'position': position,
            'exit_time': timestamp,
            'exit_price': current_price,
            'stop_loss': stop_loss,
            'pnl': trade_pnl,
            'pnl_percent': (trade_pnl / (position_size * entry_price)) * 100,
            'exit_reason': 'end_of_data',
            'win_loss': 'win' if trade_pnl > 0 else 'loss',
            'monetary_change': trade_pnl,
            'balance_before': balance - trade_pnl,
            'balance_after': balance
        })

    # Convert trades list to DataFrame and save to CSV
    trades_df = pd.DataFrame(trades) if trades else pd.DataFrame()
    if not trades_df.empty:
        trades_df.set_index('entry_time', inplace=True)
        trades_df.index = pd.to_datetime(trades_df.index)
        trades_df['entry_date'] = trades_df.index.date
        
        # Fix: Calculate days from start using list comprehension
        start_datetime = pd.Timestamp(start_date)
        trades_df['days_from_start'] = [(date - start_datetime).days for date in pd.to_datetime(trades_df.index)]
        
        # Print warmup period validation
        print("\nTrade Date Validation:")
        print(f"Start Date: {start_date}")
        print(f"Trading Start Date: {trading_start_date}")
        print("\nFirst few trades with days from start:")
        print(trades_df[['entry_date', 'days_from_start', 'position', 'win_loss', 'monetary_change']].head())

        # Verify no trades during warmup
        warmup_trades = trades_df[trades_df['days_from_start'] < 5]
        if not warmup_trades.empty:
            print("\n⚠️ Warning: Found trades during warmup period!")
            print(warmup_trades)

        # Save to CSV
        trades_df.to_csv(os.path.join(output_dir, 'trades.csv'))
        print(f"\nTrades saved to {os.path.join(output_dir, 'trades.csv')}")

    # Convert portfolio history to DataFrame
    portfolio_df = pd.DataFrame(portfolio_history)
    if not portfolio_df.empty:
        portfolio_df.set_index('timestamp', inplace=True)

    return trades_df, portfolio_df

In [12]:
# Initialize trades_df as None or empty DataFrame
trades_df = pd.DataFrame()
portfolio_df = pd.DataFrame()

# Prepare trading signals
signals_df['signal'] = 'no_signal'

# Generate long entry signals
signals_df.loc[(signals_df['trend_bias'] == 'bullish') & 
              (signals_df['price_crossed_above_ema']) & 
              (signals_df['rsi_9'] > 30), 'signal'] = 'buy'

# Generate short entry signals
signals_df.loc[(signals_df['trend_bias'] == 'bearish') & 
              (signals_df['price_crossed_below_ema']) & 
              (signals_df['rsi_9'] < 70), 'signal'] = 'sell'

# Generate exit signals
signals_df.loc[signals_df['macd_cross_below'], 'signal'] = 'exit_buy'
signals_df.loc[signals_df['macd_cross_above'], 'signal'] = 'exit_sell'

# Calculate stop losses (example: 20 pips)
pip_value = 0.001
stop_loss_pips = 10
signals_df['stop_loss_long'] = signals_df['close'] - (stop_loss_pips * pip_value)
signals_df['stop_loss_short'] = signals_df['close'] + (stop_loss_pips * pip_value)

# Execute trades with initial balance of $10,000 and 2% risk per trade
trades_df, portfolio_df = execute_trades(signals_df, initial_balance=10, position_size_percent=2)

print("\nTrade Summary:")
print(f"Number of trades executed: {len(trades_df)}")
print("\nFirst few trades:")
display(trades_df.head())

📁 Output directory created/verified: ./strategy_results
Warmup period: 2025-04-16 to 2025-04-21
🔄 Processing 2027 price data points...


TypeError: Cannot subtract tz-naive and tz-aware datetime-like objects.

In [ ]:
# Calculate and display performance metrics
performance_metrics = calculate_performance_metrics(trades_df, portfolio_df, initial_balance=10)

print("\nPerformance Metrics:")
print(f"Total Trades: {performance_metrics['total_trades']}")
print(f"Win Rate: {performance_metrics['win_rate']:.2f}%")
print(f"Net Profit: ${performance_metrics['net_profit']:.2f}")
print(f"Profit Factor: {performance_metrics['profit_factor']:.2f}")
print(f"Average Trade: ${performance_metrics['avg_trade']:.2f}")
print(f"Max Drawdown: {performance_metrics['max_drawdown_percent']:.2f}%")
print(f"Sharpe Ratio: {performance_metrics['sharpe_ratio']:.2f}")

# Generate and display performance charts
generate_performance_charts(trades_df, portfolio_df, performance_metrics, './strategy_results', 
                          datetime.now().strftime('%Y%m%d_%H%M%S'))

# Display equity curve
plt.figure(figsize=(12, 6))
portfolio_df['equity'].plot(title='Strategy Equity Curve')
plt.grid(True)
plt.show()

In [ ]:
class DynamicDataFetcher(TradingViewDataFetcher):
    def __init__(self, config, username=None, password=None):
        super().__init__(username, password)
        self.config = config
        
    def fetch_all_timeframes(self):
        """Fetch data for all configured timeframes"""
        data = {}
        for timeframe in self.config['timeframes']:
            data[timeframe] = self.fetch_historical_data(
                symbol=self.config['symbol'],
                exchange=self.config['exchange'],
                interval=timeframe,
                days_back=self.config['history_days']
            )
        return data
    
    def update_all_timeframes(self):
        """Update data for all configured timeframes"""
        for timeframe in self.config['timeframes']:
            self.update_data(
                symbol=self.config['symbol'],
                exchange=self.config['exchange'],
                interval=timeframe
            )

In [ ]:
class DynamicDataProcessor:
    def __init__(self, config):
        self.config = config
    
    def calculate_indicators(self, data):
        """Calculate indicators based on configuration"""
        processed_data = {}
        
        # Process H1 data for trend
        h1_data = data['H1'].copy()
        h1_data[f'ema_{self.config["trend_ema_period"]}'] = ta.ema(
            h1_data['close'], 
            length=self.config['trend_ema_period']
        )
        h1_data['trend_bias'] = np.where(
            h1_data['close'] > h1_data[f'ema_{self.config["trend_ema_period"]}'], 
            'bullish', 'bearish'
        )
        
        # Process M5 data for entries and exits
        m5_data = data['M5'].copy()
        
        # Calculate EMA for entry
        m5_data[f'ema_{self.config["entry_ema_period"]}'] = ta.ema(
            m5_data['close'], 
            length=self.config['entry_ema_period']
        )
        
        # Calculate RSI
        m5_data['rsi'] = ta.rsi(m5_data['close'], length=self.config['rsi_period'])
        
        # Calculate MACD
        macd = ta.macd(
            m5_data['close'], 
            fast=self.config['macd_fast'], 
            slow=self.config['macd_slow'], 
            signal=self.config['macd_signal']
        )
        m5_data = m5_data.join(macd)
        
        processed_data['H1'] = h1_data
        processed_data['M5'] = m5_data
        
        return processed_data
    
    def generate_signals(self, data):
        """Generate trading signals based on processed data"""
        m5_data = data['M5'].copy()
        h1_data = data['H1'].copy()
        
        # Resample H1 trend bias to M5 timeframe
        m5_data['trend_bias'] = h1_data['trend_bias'].reindex(
            m5_data.index, method='ffill'
        )
        
        # Generate signals
        m5_data['signal'] = 'no_signal'
        
        # Long entry conditions
        m5_data.loc[
            (m5_data['trend_bias'] == 'bullish') &
            (m5_data['close'] > m5_data[f'ema_{self.config["entry_ema_period"]}']) &
            (m5_data['rsi'] > self.config['rsi_oversold']),
            'signal'
        ] = 'buy'
        
        # Short entry conditions
        m5_data.loc[
            (m5_data['trend_bias'] == 'bearish') &
            (m5_data['close'] < m5_data[f'ema_{self.config["entry_ema_period"]}']) &
            (m5_data['rsi'] < self.config['rsi_overbought']),
            'signal'
        ] = 'sell'
        
        # Calculate stop losses
        m5_data['stop_loss_long'] = m5_data['close'] - (
            self.config['stop_loss_pips'] * self.config['pip_value']
        )
        m5_data['stop_loss_short'] = m5_data['close'] + (
            self.config['stop_loss_pips'] * self.config['pip_value']
        )
        
        return m5_data

In [ ]:
# Initialize the data processor
processor = DataProcessor()

# Example of processing cycle
def process_cycle():
    print("Starting data processing cycle...")
    results = processor.get_latest_data()
    
    for timeframe, data in results.items():
        print(f"\nLatest {timeframe} data:")
        if data is not None:
            print(data.tail())

# Run a processing cycle
process_cycle()

# For continuous processing, you could use:
'''
while True:
    process_cycle()
    time.sleep(60)  # Wait 1 minute before next check
'''